In [6]:
import csv, json
import os, grpc
from datetime import datetime

from senzing_grpc import SzAbstractFactoryGrpc

grpc_url = f"{os.getenv('SENZING_GRPC_HOST', 'senzing')}:{os.getenv('SENZING_GRPC_PORT', '8261')}"
grpc_channel = grpc.insecure_channel(grpc_url)

sz_factory = SzAbstractFactoryGrpc(grpc_channel)
sz_engine = sz_factory.create_engine()
sz_configmanager = sz_factory.create_configmanager()

print(f"✅ Connected to Senzing at {grpc_url}")

✅ Connected to Senzing at resolver:8261


In [2]:
DATA_SOURCE = 'BURLINGAME'
TSV_FILE = '/workspace/data/filtered/burlingame.tsv'

In [3]:
def map_row(row):
    features = [
        {'RECORD_TYPE': 'ORGANIZATION'},
        {'NAME_ORG': row['Business Name'].strip()},
    ]
    address = row['Address'].strip()
    if address != '--ON FILE--':          # sentinel, not a real address
        features.append({'ADDR_TYPE': 'BUSINESS', 'ADDR_FULL': address})

    return {
        'DATA_SOURCE': DATA_SOURCE,
        'RECORD_ID': row['Account #'].strip(),
        'FEATURES': features,
        # payload only - these are LICENSE dates, not incorporation dates
        'LICENSE_START_DATE': row['Start Date'].strip(),
        'LICENSE_EXPIRE_DATE': row['Expire Date'].strip(),
    }

In [4]:
with open(TSV_FILE, newline='', encoding='utf-8-sig') as f:
    records = [map_row(row) for row in csv.DictReader(f, delimiter='\t')]

print(f"✅ Mapped {len(records):,} records")
print(json.dumps(records[0], indent=2))

✅ Mapped 4,781 records
{
  "DATA_SOURCE": "BURLINGAME",
  "RECORD_ID": "20509315",
  "FEATURES": [
    {
      "RECORD_TYPE": "ORGANIZATION"
    },
    {
      "NAME_ORG": "101 MOBILITY"
    },
    {
      "ADDR_TYPE": "BUSINESS",
      "ADDR_FULL": "1731 ADRIAN RD STE 9, BURLINGAME, CA 94010-2109"
    }
  ],
  "LICENSE_START_DATE": "7/1/2025",
  "LICENSE_EXPIRE_DATE": "6/30/2027"
}


In [8]:
sz_config = sz_configmanager.create_config_from_config_id(
    sz_configmanager.get_default_config_id())
registry = json.loads(sz_config.get_data_source_registry())

if DATA_SOURCE not in [ds['DSRC_CODE'] for ds in registry['DATA_SOURCES']]:
    sz_config.register_data_source(DATA_SOURCE)
    new_id = sz_configmanager.register_config(sz_config.export(), f"Added {DATA_SOURCE}")
    sz_configmanager.set_default_config_id(new_id)
    print(f"✅ Registered {DATA_SOURCE} - now run: docker restart erkg_senzing")
else:
    print(f"✅ {DATA_SOURCE} already registered")

✅ Registered BURLINGAME - now run: docker restart erkg_senzing


In [9]:
records_loaded, errors = 0, []

for record in records:
    try:
        sz_engine.add_record(DATA_SOURCE, record['RECORD_ID'], json.dumps(record))
        records_loaded += 1
    except Exception as err:
        errors.append(f"{record['RECORD_ID']}: {err}")

print(f"✅ Loaded {records_loaded:,} records, {len(errors)} errors")


✅ Loaded 4,781 records, 0 errors


In [17]:
DATA_SOURCE = 'ABC-RETAIL'
CSV_FILE = '/workspace/data/filtered/CA-ABC-LicenseReport-retail.csv'

In [18]:
def map_row(row):
    # 13 rows have no business name - fall back to the owner
    name = row['Business Name'].strip() or row['Primary Owner'].strip()
    features = [
        {'RECORD_TYPE': 'ORGANIZATION'},
        {'NAME_ORG': name},
    ]

    # "80 NEW PLACE RD,HILLSBOROUGH, CA  94010Census Tract:  6056.00"
    address = row['Premises Addr.'].split('Census Tract')[0].strip().rstrip(',')
    if address:
        features.append({'ADDR_TYPE': 'BUSINESS', 'ADDR_FULL': address})

    return {
        'DATA_SOURCE': DATA_SOURCE,
        # license number alone is NOT unique - same premises, several license types
        'RECORD_ID': f"{row['License Number'].strip()}-{row['License Type'].strip()}",
        'FEATURES': features,
        'OWNER_NAME': row['Primary Owner'].strip(),
        'LICENSE_NUMBER': row['License Number'].strip(),
        'LICENSE_TYPE': row['License Type'].strip(),
        'LICENSE_STATUS': row['Status'].strip(),
        'LICENSE_ISSUE_DATE': row['Orig. Iss. Date'].strip(),
        'LICENSE_EXPIRE_DATE': row['Expir. Date'].strip(),
    }

In [19]:
with open(CSV_FILE, newline='', encoding='utf-8-sig') as f:
    records = [map_row(row) for row in csv.DictReader(f)]

print(f"✅ Mapped {len(records):,} records")
print(json.dumps(records[0], indent=2))

✅ Mapped 201 records
{
  "DATA_SOURCE": "ABC-RETAIL",
  "RECORD_ID": "4693-51",
  "FEATURES": [
    {
      "RECORD_TYPE": "ORGANIZATION"
    },
    {
      "NAME_ORG": "BURLINGAME COUNTRY CLUB"
    },
    {
      "ADDR_TYPE": "BUSINESS",
      "ADDR_FULL": "80 NEW PLACE RD,HILLSBOROUGH, CA  94010"
    }
  ],
  "OWNER_NAME": "BURLINGAME COUNTRY CLUB",
  "LICENSE_NUMBER": "4693",
  "LICENSE_TYPE": "51",
  "LICENSE_STATUS": "ACTIVE",
  "LICENSE_ISSUE_DATE": "01/03/1989",
  "LICENSE_EXPIRE_DATE": "03/31/2027"
}


In [22]:
sz_config = sz_configmanager.create_config_from_config_id(
    sz_configmanager.get_default_config_id())
registry = json.loads(sz_config.get_data_source_registry())

if DATA_SOURCE not in [ds['DSRC_CODE'] for ds in registry['DATA_SOURCES']]:
    sz_config.register_data_source(DATA_SOURCE)
    new_id = sz_configmanager.register_config(sz_config.export(), f"Added {DATA_SOURCE}")
    sz_configmanager.set_default_config_id(new_id)
    print(f"✅ Registered {DATA_SOURCE} - now run: docker restart erkg_senzing")
else:
    print(f"✅ {DATA_SOURCE} already registered")

✅ Registered ABC-RETAIL - now run: docker restart erkg_senzing


`docker restart erkg_senzing` on error

In [24]:
records_loaded, errors = 0, []

for record in records:
    try:
        sz_engine.add_record(DATA_SOURCE, record['RECORD_ID'], json.dumps(record))
        records_loaded += 1
    except Exception as err:
        errors.append(f"{record['RECORD_ID']}: {err}")

print(f"✅ Loaded {records_loaded:,} records, {len(errors)} errors")

✅ Loaded 201 records, 0 errors


In [25]:
DATA_SOURCE = 'ABC-NONRETAIL'
CSV_FILE = '/workspace/data/filtered/CA-ABC-LicenseReport-nonretail.csv'

def map_row(row):
    # 5 rows have no business name - fall back to the owner
    name = row['Business Name'].strip() or row['Primary Owner'].strip()
    features = [
        {'RECORD_TYPE': 'ORGANIZATION'},
        {'NAME_ORG': name},
    ]

    address = row['Premises Addr.'].split('Census Tract')[0].strip().rstrip(',')
    if address:
        features.append({'ADDR_TYPE': 'BUSINESS', 'ADDR_FULL': address})

    return {
        'DATA_SOURCE': DATA_SOURCE,
        # license number alone is NOT unique - same owner, several license types
        'RECORD_ID': f"{row['License Number'].strip()}-{row['License Type'].strip()}",
        'FEATURES': features,
        'OWNER_NAME': row['Primary Owner'].strip(),
        'LICENSE_NUMBER': row['License Number'].strip(),
        'LICENSE_TYPE': row['License Type'].strip(),
        'LICENSE_STATUS': row['Status'].strip(),
        'LICENSE_ISSUE_DATE': row['Orig. Iss. Date'].strip(),
        'LICENSE_EXPIRE_DATE': row['Expir. Date'].strip(),
    }

with open(CSV_FILE, newline='', encoding='utf-8-sig') as f:
    records = [map_row(row) for row in csv.DictReader(f)]

print(f"✅ Mapped {len(records):,} records")
print(json.dumps(records[0], indent=2))

✅ Mapped 19 records
{
  "DATA_SOURCE": "ABC-NONRETAIL",
  "RECORD_ID": "358480-13",
  "FEATURES": [
    {
      "RECORD_TYPE": "ORGANIZATION"
    },
    {
      "NAME_ORG": "INTERNATIONAL BEVERAGE"
    },
    {
      "ADDR_TYPE": "BUSINESS",
      "ADDR_FULL": "330 PRIMROSE RD, STE 512,BURLINGAME, CA  94010"
    }
  ],
  "OWNER_NAME": "ALLIED LOMAR INC",
  "LICENSE_NUMBER": "358480",
  "LICENSE_TYPE": "13",
  "LICENSE_STATUS": "ACTIVE",
  "LICENSE_ISSUE_DATE": "02/10/2000",
  "LICENSE_EXPIRE_DATE": "01/31/2027"
}


In [26]:
sz_config = sz_configmanager.create_config_from_config_id(
    sz_configmanager.get_default_config_id())
registry = json.loads(sz_config.get_data_source_registry())

if DATA_SOURCE not in [ds['DSRC_CODE'] for ds in registry['DATA_SOURCES']]:
    sz_config.register_data_source(DATA_SOURCE)
    new_id = sz_configmanager.register_config(sz_config.export(), f"Added {DATA_SOURCE}")
    sz_configmanager.set_default_config_id(new_id)
    print(f"✅ Registered {DATA_SOURCE} - now run: docker restart erkg_senzing")
else:
    print(f"✅ {DATA_SOURCE} already registered")

✅ Registered ABC-NONRETAIL - now run: docker restart erkg_senzing


In [28]:
records_loaded, errors = 0, []

for record in records:
    try:
        sz_engine.add_record(DATA_SOURCE, record['RECORD_ID'], json.dumps(record))
        records_loaded += 1
    except Exception as err:
        errors.append(f"{record['RECORD_ID']}: {err}")

print(f"✅ Loaded {records_loaded:,} records, {len(errors)} errors")

✅ Loaded 19 records, 0 errors


In [29]:
DATA_SOURCE = 'CA-SOS-AGENTS'
DATA_FILE = '/workspace/data/filtered/Agents.csv'

def drop_empty(d):
    return {k: v for k, v in d.items() if v}

def map_row(row):
    # ORG_NAME and LAST_NAME are mutually exclusive - the agent is a company or a person
    if row['ORG_NAME'].strip():
        record_type = 'ORGANIZATION'
        name = {'NAME_ORG': row['ORG_NAME'].strip()}
    else:
        record_type = 'PERSON'
        name = drop_empty({
            'NAME_FIRST': row['FIRST_NAME'].strip(),
            'NAME_MIDDLE': row['MIDDLE_NAME'].strip(),
            'NAME_LAST': row['LAST_NAME'].strip(),
        })

    features = [{'RECORD_TYPE': record_type}, name]

    # this file gives us parsed address components - Senzing prefers those over ADDR_FULL
    address = drop_empty({
        'ADDR_TYPE': 'BUSINESS',
        'ADDR_LINE1': row['PHYSICAL_ADDRESS1'].strip(),
        'ADDR_LINE2': row['PHYSICAL_ADDRESS2'].strip(),
        'ADDR_CITY': row['PHYSICAL_CITY'].strip(),
        'ADDR_STATE': row['PHYSICAL_STATE'].strip(),
        'ADDR_POSTAL_CODE': row['PHYSICAL_POSTAL_CODE'].strip(),
        'ADDR_COUNTRY': row['PHYSICAL_COUNTRY'].strip(),
    })
    if 'ADDR_LINE1' in address:
        features.append(address)

    return {
        'DATA_SOURCE': DATA_SOURCE,
        'RECORD_ID': row['ENTITY_NUM'].strip(),
        'FEATURES': features,
        'AGENT_TYPE': row['AGENT_TYPE'].strip(),
        'REPRESENTS_COMPANY': row['ENTITY_NAME'].strip(),
    }

# multi-character '*|*' delimiter - csv.DictReader only takes a single char
lines = open(DATA_FILE, encoding='utf-8-sig').read().splitlines()
header = lines[0].split('*|*')
records = [map_row(dict(zip(header, line.split('*|*'))))
           for line in lines[1:] if line.strip()]

print(f"✅ Mapped {len(records):,} records")
print(json.dumps(records[0], indent=2))

✅ Mapped 7,421 records
{
  "DATA_SOURCE": "CA-SOS-AGENTS",
  "RECORD_ID": "B20260056953",
  "FEATURES": [
    {
      "RECORD_TYPE": "PERSON"
    },
    {
      "NAME_FIRST": "Richard",
      "NAME_MIDDLE": "Hugh",
      "NAME_LAST": "Busse"
    },
    {
      "ADDR_TYPE": "BUSINESS",
      "ADDR_LINE1": "610 TRAYER AVE",
      "ADDR_CITY": "GLENDORA",
      "ADDR_STATE": "CA",
      "ADDR_POSTAL_CODE": "91741",
      "ADDR_COUNTRY": "UNITED STATES"
    }
  ],
  "AGENT_TYPE": "Individual Agent",
  "REPRESENTS_COMPANY": "Busse Manufacturing LLC"
}


In [30]:
sz_config = sz_configmanager.create_config_from_config_id(
    sz_configmanager.get_default_config_id())
registry = json.loads(sz_config.get_data_source_registry())

if DATA_SOURCE not in [ds['DSRC_CODE'] for ds in registry['DATA_SOURCES']]:
    sz_config.register_data_source(DATA_SOURCE)
    new_id = sz_configmanager.register_config(sz_config.export(), f"Added {DATA_SOURCE}")
    sz_configmanager.set_default_config_id(new_id)
    print(f"✅ Registered {DATA_SOURCE} - now run: docker restart erkg_senzing")
else:
    print(f"✅ {DATA_SOURCE} already registered")

✅ Registered CA-SOS-AGENTS - now run: docker restart erkg_senzing


In [32]:
records_loaded, errors = 0, []

for record in records:
    try:
        sz_engine.add_record(DATA_SOURCE, record['RECORD_ID'], json.dumps(record))
        records_loaded += 1
    except Exception as err:
        errors.append(f"{record['RECORD_ID']}: {err}")

print(f"✅ Loaded {records_loaded:,} records, {len(errors)} errors")

✅ Loaded 7,421 records, 0 errors


In [34]:
from datetime import datetime

DATA_SOURCE = 'CA-SOS-FILINGS'
DATA_FILE = '/workspace/data/filtered/Filings.csv'

def drop_empty(d):
    return {k: v for k, v in d.items() if v}

def to_iso(value):
    try:
        return datetime.strptime(value.strip(), '%m/%d/%Y').strftime('%Y-%m-%d')
    except ValueError:
        return ''

def address(row, prefix, addr_type, street='ADDRESS'):
    a = drop_empty({
        'ADDR_TYPE': addr_type,
        'ADDR_LINE1': row.get(f'{prefix}{street}', '').strip(),
        'ADDR_LINE2': row.get(f'{prefix}{street}2', '').strip(),
        'ADDR_CITY': row.get(f'{prefix}CITY', '').strip(),
        'ADDR_STATE': row.get(f'{prefix}STATE', '').strip(),
        'ADDR_POSTAL_CODE': row.get(f'{prefix}POSTAL_CODE', '').strip(),
        'ADDR_COUNTRY': row.get(f'{prefix}COUNTRY', '').strip(),
    })
    return a if 'ADDR_LINE1' in a else None

def map_row(row):
    features = [
        {'RECORD_TYPE': 'ORGANIZATION'},
        {'NAME_ORG': row['ENTITY_NAME'].strip()},
    ]

    # 25 rows carry an alternate name the company trades under
    if row['FOREIGN_NAME'].strip():
        features.append({'NAME_TYPE': 'DBA', 'NAME_ORG': row['FOREIGN_NAME'].strip()})

    # unlike the license files, THIS really is an incorporation date
    filed = to_iso(row['INITIAL_FILING_DATE'])
    if filed:
        features.append({'REGISTRATION_DATE': filed})

    for a in (address(row, 'PRINCIPAL_', 'BUSINESS'),
              address(row, 'MAILING_', 'MAILING'),
              address(row, 'PRINCIPAL_', 'BUSINESS', 'ADDRESS_IN_CA')):
        if a:
            features.append(a)

    return {
        'DATA_SOURCE': DATA_SOURCE,
        'RECORD_ID': row['ENTITY_NUM'].strip(),
        'FEATURES': features,
        'ENTITY_TYPE': row['ENTITY_TYPE'].strip(),
        'ENTITY_STATUS': row['ENTITY_STATUS'].strip(),
        'JURISDICTION': row['JURISDICTION'].strip(),
        'TYPE_OF_BUSINESS': row['TYPE_OF_BUSINESS'].strip(),
    }

lines = open(DATA_FILE, encoding='utf-8-sig').read().splitlines()
header = lines[0].split('*|*')
records = [map_row(dict(zip(header, line.split('*|*'))))
           for line in lines[1:] if line.strip()]

print(f"✅ Mapped {len(records):,} records")
print(json.dumps(records[0], indent=2))


✅ Mapped 7,801 records
{
  "DATA_SOURCE": "CA-SOS-FILINGS",
  "RECORD_ID": "5920085",
  "FEATURES": [
    {
      "RECORD_TYPE": "ORGANIZATION"
    },
    {
      "NAME_ORG": "Polo's Dental Laboratory, LLC"
    },
    {
      "REGISTRATION_DATE": "2026-07-15"
    }
  ],
  "ENTITY_TYPE": "Name Reservation",
  "ENTITY_STATUS": "Active",
  "JURISDICTION": "CALIFORNIA",
  "TYPE_OF_BUSINESS": ""
}


In [35]:
sz_config = sz_configmanager.create_config_from_config_id(
    sz_configmanager.get_default_config_id())
registry = json.loads(sz_config.get_data_source_registry())

if DATA_SOURCE not in [ds['DSRC_CODE'] for ds in registry['DATA_SOURCES']]:
    sz_config.register_data_source(DATA_SOURCE)
    new_id = sz_configmanager.register_config(sz_config.export(), f"Added {DATA_SOURCE}")
    sz_configmanager.set_default_config_id(new_id)
    print(f"✅ Registered {DATA_SOURCE} - now run: docker restart erkg_senzing")
else:
    print(f"✅ {DATA_SOURCE} already registered")

✅ Registered CA-SOS-FILINGS - now run: docker restart erkg_senzing


In [37]:
records_loaded, errors = 0, []

for record in records:
    try:
        sz_engine.add_record(DATA_SOURCE, record['RECORD_ID'], json.dumps(record))
        records_loaded += 1
    except Exception as err:
        errors.append(f"{record['RECORD_ID']}: {err}")

print(f"✅ Loaded {records_loaded:,} records, {len(errors)} errors")

✅ Loaded 7,801 records, 0 errors


In [38]:
DATA_SOURCE = 'CA-SOS-PRINCIPALS'
DATA_FILE = '/workspace/data/filtered/Principals.csv'

def drop_empty(d):
    return {k: v for k, v in d.items() if v}

def map_row(row, seq):
    # ORG_NAME and LAST_NAME are mutually exclusive, same as Agents.csv
    if row['ORG_NAME'].strip():
        record_type = 'ORGANIZATION'
        name = {'NAME_ORG': row['ORG_NAME'].strip()}
    else:
        record_type = 'PERSON'
        name = drop_empty({
            'NAME_FIRST': row['FIRST_NAME'].strip(),
            'NAME_MIDDLE': row['MIDDLE_NAME'].strip(),
            'NAME_LAST': row['LAST_NAME'].strip(),
        })

    features = [{'RECORD_TYPE': record_type}, name]

    address = drop_empty({
        'ADDR_TYPE': 'BUSINESS',
        'ADDR_LINE1': row['ADDRESS1'].strip(),
        'ADDR_LINE2': row['ADDRESS2'].strip(),
        'ADDR_CITY': row['CITY'].strip(),
        'ADDR_STATE': row['STATE'].strip(),
        'ADDR_POSTAL_CODE': row['POSTAL_CODE'].strip(),
        'ADDR_COUNTRY': row['COUNTRY'].strip(),
    })
    if 'ADDR_LINE1' in address:
        features.append(address)

    return {
        'DATA_SOURCE': DATA_SOURCE,
        # a company has up to 12 principals, so ENTITY_NUM alone is not a key
        'RECORD_ID': f"{row['ENTITY_NUM'].strip()}-P{seq}",
        'FEATURES': features,
        'POSITION_TYPE': row['POSITION_TYPE'].strip(),
        'COMPANY_ENTITY_NUM': row['ENTITY_NUM'].strip(),
        'COMPANY_NAME': row['ENTITY_NAME'].strip(),
    }

lines = open(DATA_FILE, encoding='utf-8-sig').read().splitlines()
header = lines[0].split('*|*')
records = [map_row(dict(zip(header, line.split('*|*'))), seq)
           for seq, line in enumerate(l for l in lines[1:] if l.strip())]

print(f"✅ Mapped {len(records):,} records")
print(json.dumps(records[0], indent=2))


✅ Mapped 5,701 records
{
  "DATA_SOURCE": "CA-SOS-PRINCIPALS",
  "RECORD_ID": "B20260299459-P0",
  "FEATURES": [
    {
      "RECORD_TYPE": "PERSON"
    },
    {
      "NAME_FIRST": "Armann",
      "NAME_LAST": "Ghazaryan"
    },
    {
      "ADDR_TYPE": "BUSINESS",
      "ADDR_LINE1": "10046 SAMOA AVE",
      "ADDR_LINE2": "414",
      "ADDR_CITY": "TUJUNGA",
      "ADDR_STATE": "CA",
      "ADDR_POSTAL_CODE": "91042",
      "ADDR_COUNTRY": "United States"
    }
  ],
  "POSITION_TYPE": "Director",
  "COMPANY_ENTITY_NUM": "B20260299459",
  "COMPANY_NAME": "Arahet Group Inc"
}


In [39]:
sz_config = sz_configmanager.create_config_from_config_id(
    sz_configmanager.get_default_config_id())
registry = json.loads(sz_config.get_data_source_registry())

if DATA_SOURCE not in [ds['DSRC_CODE'] for ds in registry['DATA_SOURCES']]:
    sz_config.register_data_source(DATA_SOURCE)
    new_id = sz_configmanager.register_config(sz_config.export(), f"Added {DATA_SOURCE}")
    sz_configmanager.set_default_config_id(new_id)
    print(f"✅ Registered {DATA_SOURCE} - now run: docker restart erkg_senzing")
else:
    print(f"✅ {DATA_SOURCE} already registered")

✅ Registered CA-SOS-PRINCIPALS - now run: docker restart erkg_senzing


In [41]:
records_loaded, errors = 0, []

for record in records:
    try:
        sz_engine.add_record(DATA_SOURCE, record['RECORD_ID'], json.dumps(record))
        records_loaded += 1
    except Exception as err:
        errors.append(f"{record['RECORD_ID']}: {err}")

print(f"✅ Loaded {records_loaded:,} records, {len(errors)} errors")

✅ Loaded 5,701 records, 0 errors


In [42]:
DATA_SOURCE = 'CA-CONTRACTORS'
CSV_FILE = '/workspace/data/filtered/master-list-of-ca-licensed-contractors.csv'

def drop_empty(d):
    return {k: v for k, v in d.items() if v}

def map_row(row):
    features = [
        {'RECORD_TYPE': 'ORGANIZATION'},
        {'NAME_ORG': row['BusinessName'].strip()},
    ]

    # up to two more trade names - skip any that repeat the primary
    seen = {row['BusinessName'].strip().upper()}
    for col in ('FullBusinessName', 'BUS-NAME-2'):
        alt = row[col].strip()
        if alt and alt.upper() not in seen:
            features.append({'NAME_TYPE': 'DBA', 'NAME_ORG': alt})
            seen.add(alt.upper())

    address = drop_empty({
        'ADDR_TYPE': 'BUSINESS',
        'ADDR_LINE1': row['MailingAddress'].strip(),
        'ADDR_CITY': row['City'].strip(),
        'ADDR_STATE': row['State'].strip(),
        'ADDR_POSTAL_CODE': row['ZIPCode'].strip(),
    })
    if 'ADDR_LINE1' in address:
        features.append(address)

    phone = row['BusinessPhone'].strip()
    if phone:
        features.append({'PHONE_TYPE': 'WORK', 'PHONE_NUMBER': phone})

    return {
        'DATA_SOURCE': DATA_SOURCE,
        'RECORD_ID': row['LicenseNo'].strip(),
        'FEATURES': features,
        'BUSINESS_TYPE': row['BusinessType'].strip(),
        'COUNTY': row['County'].strip(),
        'PRIMARY_STATUS': row['PrimaryStatus'].strip(),
        'CLASSIFICATIONS': row['Classifications(s)'].strip(),
        'LICENSE_ISSUE_DATE': row['IssueDate'].strip(),
        'LICENSE_EXPIRE_DATE': row['ExpirationDate'].strip(),
    }

with open(CSV_FILE, newline='', encoding='utf-8-sig') as f:
    records = [map_row(row) for row in csv.DictReader(f)]

print(f"✅ Mapped {len(records):,} records")
print(json.dumps(records[0], indent=2))


✅ Mapped 3,274 records
{
  "DATA_SOURCE": "CA-CONTRACTORS",
  "RECORD_ID": "1000087",
  "FEATURES": [
    {
      "RECORD_TYPE": "ORGANIZATION"
    },
    {
      "NAME_ORG": "CATO'S PAVING"
    },
    {
      "NAME_TYPE": "DBA",
      "NAME_ORG": "CATO'S GENERAL ENGINEERING INC"
    },
    {
      "ADDR_TYPE": "BUSINESS",
      "ADDR_LINE1": "22302 HATHAWAY AVE",
      "ADDR_CITY": "HAYWARD",
      "ADDR_STATE": "CA",
      "ADDR_POSTAL_CODE": "94541"
    },
    {
      "PHONE_TYPE": "WORK",
      "PHONE_NUMBER": "(510) 397 2677"
    }
  ],
  "BUSINESS_TYPE": "Corporation",
  "COUNTY": "Alameda",
  "PRIMARY_STATUS": "CLEAR",
  "CLASSIFICATIONS": "A",
  "LICENSE_ISSUE_DATE": "01/12/2015",
  "LICENSE_EXPIRE_DATE": "01/31/2027"
}


In [43]:
sz_config = sz_configmanager.create_config_from_config_id(
    sz_configmanager.get_default_config_id())
registry = json.loads(sz_config.get_data_source_registry())

if DATA_SOURCE not in [ds['DSRC_CODE'] for ds in registry['DATA_SOURCES']]:
    sz_config.register_data_source(DATA_SOURCE)
    new_id = sz_configmanager.register_config(sz_config.export(), f"Added {DATA_SOURCE}")
    sz_configmanager.set_default_config_id(new_id)
    print(f"✅ Registered {DATA_SOURCE} - now run: docker restart erkg_senzing")
else:
    print(f"✅ {DATA_SOURCE} already registered")

✅ Registered CA-CONTRACTORS - now run: docker restart erkg_senzing


In [45]:
records_loaded, errors = 0, []

for record in records:
    try:
        sz_engine.add_record(DATA_SOURCE, record['RECORD_ID'], json.dumps(record))
        records_loaded += 1
    except Exception as err:
        errors.append(f"{record['RECORD_ID']}: {err}")

print(f"✅ Loaded {records_loaded:,} records, {len(errors)} errors")

✅ Loaded 3,274 records, 0 errors


In [46]:
import re

DATA_SOURCE = 'CA-CONTRACTOR-PERSONNEL'
CSV_FILE = '/workspace/data/filtered/master-list-of-ca-licensed-contractors-personnel.csv'

def drop_empty(d):
    return {k: v for k, v in d.items() if v}

def name_feature(raw, kind, is_first):
    """'DAVIS      DANA      MICHAEL' -> parsed name. Fields are space-padded."""
    raw = raw.strip()
    if not raw:
        return None

    name_type = 'PRIMARY' if is_first else ('DBA' if kind == 'Business' else 'AKA')

    if kind == 'Business':
        return {'NAME_TYPE': name_type, 'NAME_ORG': ' '.join(raw.split())}

    parts = re.split(r'\s{2,}', raw)          # 2+ spaces, so "MC CURDY" stays intact
    f = {'NAME_TYPE': name_type, 'NAME_LAST': parts[0].strip()}
    if len(parts) > 1:
        f['NAME_FIRST'] = parts[1].strip()
    if len(parts) > 2:
        f['NAME_MIDDLE'] = parts[2].strip()
    if len(parts) > 3:
        f['NAME_SUFFIX'] = ' '.join(p.strip() for p in parts[3:])
    return drop_empty(f)

def map_row(row):
    # 385 rows pack two names into one field: "PRIMARY NAME| AKA NAME"
    names = row['Name'].split('|')
    kinds = [k.strip() for k in row['Name-TP'].split('|')]
    kinds += [kinds[-1]] * (len(names) - len(kinds))   # pad if fewer types than names

    record_type = 'ORGANIZATION' if kinds[0] == 'Business' else 'PERSON'
    features = [{'RECORD_TYPE': record_type}]
    for i, (n, k) in enumerate(zip(names, kinds)):
        f = name_feature(n, k, i == 0)
        if f:
            features.append(f)

    return {
        'DATA_SOURCE': DATA_SOURCE,
        # LIC-NO repeats (up to a dozen people per licence); SEQ-NO makes it unique
        'RECORD_ID': f"{row['LIC-NO'].strip()}-{row['SEQ-NO'].strip()}",
        'FEATURES': features,
        'CONTRACTOR_LICENSE_NO': row['LIC-NO'].strip(),
        'TITLES': ' '.join(row['EMP-Titl-CDE'].split()),
        'ASSOCIATION_DATE': ' '.join(row['ASSN-DT'].split()),
        'DISASSOCIATION_DATE': ' '.join(row['DIS-ASSN-DT'].split()),
    }

with open(CSV_FILE, newline='', encoding='utf-8-sig') as f:
    records = [map_row(row) for row in csv.DictReader(f)]

print(f"✅ Mapped {len(records):,} records")
print(json.dumps(records[0], indent=2))


✅ Mapped 10,215 records
{
  "DATA_SOURCE": "CA-CONTRACTOR-PERSONNEL",
  "RECORD_ID": "8-728121",
  "FEATURES": [
    {
      "RECORD_TYPE": "PERSON"
    },
    {
      "NAME_TYPE": "PRIMARY",
      "NAME_LAST": "DAVIS",
      "NAME_FIRST": "DANA",
      "NAME_MIDDLE": "MICHAEL"
    }
  ],
  "CONTRACTOR_LICENSE_NO": "8",
  "TITLES": "Officer| Officer",
  "ASSOCIATION_DATE": "03/06/2002| 11/06/2018",
  "DISASSOCIATION_DATE": "04/04/2007| 12/26/2023"
}


In [47]:
sz_config = sz_configmanager.create_config_from_config_id(
    sz_configmanager.get_default_config_id())
registry = json.loads(sz_config.get_data_source_registry())

if DATA_SOURCE not in [ds['DSRC_CODE'] for ds in registry['DATA_SOURCES']]:
    sz_config.register_data_source(DATA_SOURCE)
    new_id = sz_configmanager.register_config(sz_config.export(), f"Added {DATA_SOURCE}")
    sz_configmanager.set_default_config_id(new_id)
    print(f"✅ Registered {DATA_SOURCE} - now run: docker restart erkg_senzing")
else:
    print(f"✅ {DATA_SOURCE} already registered")

✅ Registered CA-CONTRACTOR-PERSONNEL - now run: docker restart erkg_senzing


In [49]:
records_loaded, errors = 0, []

for record in records:
    try:
        sz_engine.add_record(DATA_SOURCE, record['RECORD_ID'], json.dumps(record))
        records_loaded += 1
    except Exception as err:
        errors.append(f"{record['RECORD_ID']}: {err}")

print(f"✅ Loaded {records_loaded:,} records, {len(errors)} errors")

✅ Loaded 10,215 records, 0 errors


In [50]:
import json
import pandas as pd
from senzing import SzEngineFlags

rows, relations = [], []

handle = sz_engine.export_json_entity_report(SzEngineFlags.SZ_EXPORT_DEFAULT_FLAGS)
try:
    while True:
        line = sz_engine.fetch_next(handle)
        if not line:
            break
        entity = json.loads(line)
        resolved = entity['RESOLVED_ENTITY']

        for rec in resolved.get('RECORDS', []):
            rows.append({
                'entity_id': resolved['ENTITY_ID'],
                'entity_name': resolved.get('ENTITY_NAME', ''),
                'data_source': rec['DATA_SOURCE'],
                'record_id': rec['RECORD_ID'],
                'match_key': rec.get('MATCH_KEY', ''),
                'errule_code': rec.get('ERRULE_CODE', ''),
            })

        for rel in entity.get('RELATED_ENTITIES', []):
            relations.append({
                'entity_id': resolved['ENTITY_ID'],
                'related_id': rel['ENTITY_ID'],
                'match_level': rel.get('MATCH_LEVEL_CODE', ''),
                'match_key': rel.get('MATCH_KEY', ''),
            })
finally:
    sz_engine.close_export_report(handle)     # NOT close_export

df = pd.DataFrame(rows)
rel_df = pd.DataFrame(relations)
print(f"✅ Exported {len(df):,} records in {df.entity_id.nunique():,} entities")


✅ Exported 39,413 records in 31,339 entities


In [51]:
def banner(title):
    print("\n" + "=" * 70); print(title); print("=" * 70)

banner("DATA SOURCE SUMMARY")
summary = df.groupby('data_source').agg(
    records=('record_id', 'count'), entities=('entity_id', 'nunique'))
summary['compression'] = (summary.records / summary.entities).round(2)
print(summary.sort_values('records', ascending=False).to_string())
print(f"\nTOTAL: {len(df):,} records -> {df.entity_id.nunique():,} entities "
      f"(compression {len(df) / df.entity_id.nunique():.2f})")

banner("ENTITY SIZE BREAKDOWN")
sizes = df.groupby('entity_id').size()
buckets = pd.cut(sizes, [0, 1, 2, 5, 10, 50, 10**9],
                 labels=['1 (singleton)', '2 (pair)', '3-5', '6-10', '11-50', '51+ (review)'])
print(buckets.value_counts().sort_index().to_string())

banner("CROSS-SOURCE OVERLAP (entities spanning two sources)")
pairs = df[['entity_id', 'data_source']].drop_duplicates()
both = pairs.merge(pairs, on='entity_id')
both = both[both.data_source_x < both.data_source_y]
print(both.groupby(['data_source_x', 'data_source_y']).entity_id
          .nunique().sort_values(ascending=False).head(15).to_string())

banner("TOP MATCH KEYS (what drove resolution)")
print(df[df.match_key != ''].match_key.value_counts().head(10).to_string())

banner("LARGEST ENTITIES (review for over-matching)")
for entity_id, n in sizes.sort_values(ascending=False).head(10).items():
    e = df[df.entity_id == entity_id]
    sources = ', '.join(sorted(e.data_source.unique()))
    print(f"  {n:4d} records  {e.entity_name.iloc[0][:42]:<42} [{sources}]")

if not rel_df.empty:
    banner("RELATIONSHIPS BY MATCH LEVEL")
    print(rel_df.match_level.value_counts().to_string())
    print("\n  PM = Possibly Same, AM = Ambiguous, DR = Disclosed, PR = Possible Relation")



DATA SOURCE SUMMARY
                         records  entities  compression
data_source                                            
CA-CONTRACTOR-PERSONNEL    10215      9457         1.08
CA-SOS-FILINGS              7801      7730         1.01
CA-SOS-AGENTS               7421      4958         1.50
CA-SOS-PRINCIPALS           5701      3306         1.72
BURLINGAME                  4781      4751         1.01
CA-CONTRACTORS              3274      3186         1.03
ABC-RETAIL                   201       173         1.16
ABC-NONRETAIL                 19        14         1.36

TOTAL: 39,413 records -> 31,339 entities (compression 1.26)

ENTITY SIZE BREAKDOWN
1 (singleton)    27546
2 (pair)          2641
3-5               1081
6-10                41
11-50               22
51+ (review)         8

CROSS-SOURCE OVERLAP (entities spanning two sources)
data_source_x   data_source_y    
CA-SOS-AGENTS   CA-SOS-PRINCIPALS    1089
BURLINGAME      CA-CONTRACTORS       1037
ABC-RETAIL      BURLINGAM

In [52]:
pairs = rel_df.apply(lambda r: tuple(sorted((r.entity_id, r.related_id))), axis=1)
print(f"{pairs.nunique():,} unique related pairs")

11,822 unique related pairs


In [53]:
rel_df['pair'] = rel_df.apply(lambda r: tuple(sorted((r.entity_id, r.related_id))), axis=1)
pairs_df = rel_df.drop_duplicates('pair').drop(columns='pair')
print(pairs_df.match_level.value_counts().to_string())

match_level
POSSIBLY_RELATED    9328
POSSIBLY_SAME       2494


In [2]:
import os, psycopg2

PG = dict(host=os.getenv('POSTGRES_HOST', 'postgres'),
          port=os.getenv('POSTGRES_PORT', '5432'),
          user=os.getenv('POSTGRES_USER', 'postgres'),
          password=os.getenv('POSTGRES_PASSWORD', 'workshop'),
          dbname=os.getenv('POSTGRES_DB', 'erkg'))

SCHEMA = """
CREATE SCHEMA IF NOT EXISTS workshop;

-- Placekey cache: one row per distinct address. Re-running this notebook
-- never re-spends API quota, because only addresses missing here get sent.
CREATE TABLE IF NOT EXISTS workshop.geo_address (
    street       text NOT NULL,
    city         text NOT NULL,
    region       text,
    postal_code  text,
    placekey     text,
    latitude     double precision,
    longitude    double precision,
    looked_up_at timestamptz DEFAULT now(),
    PRIMARY KEY (street, city)
);

-- Query surface: an entity can sit at several addresses, so several rows.
CREATE TABLE IF NOT EXISTS workshop.entity_geo (
    entity_id bigint NOT NULL,
    street    text NOT NULL,
    city      text NOT NULL,
    placekey  text,
    latitude  double precision,
    longitude double precision,
    PRIMARY KEY (entity_id, street, city)
);
CREATE INDEX IF NOT EXISTS idx_entity_geo_entity ON workshop.entity_geo (entity_id);
"""

with psycopg2.connect(**PG) as conn, conn.cursor() as cur:
    cur.execute(SCHEMA)
print("✅ workshop schema ready")


✅ workshop schema ready


In [3]:
import csv

LOCAL_CITIES = {'BURLINGAME', 'MILLBRAE', 'HILLSBOROUGH'}
DATA = '/workspace/data/filtered'

def split_one_line(address):
    """'1731 ADRIAN RD STE 9, BURLINGAME, CA 94010-2109' -> components"""
    parts = address.split(',')
    if len(parts) < 3:
        return None
    state_zip = parts[-1].split()
    if len(state_zip) != 2:
        return None
    return {'street': ','.join(parts[:-2]).strip(), 'city': parts[-2].strip(),
            'region': state_zip[0], 'postal_code': state_zip[1]}

record_addresses = []

with open(f'{DATA}/burlingame.tsv', newline='', encoding='utf-8-sig') as f:
    for row in csv.DictReader(f, delimiter='\t'):
        a = split_one_line(row['Address'].strip())
        if a:
            record_addresses.append(('BURLINGAME', row['Account #'].strip(), a))

for source, fname in (('ABC-RETAIL', 'CA-ABC-LicenseReport-retail.csv'),
                      ('ABC-NONRETAIL', 'CA-ABC-LicenseReport-nonretail.csv')):
    with open(f'{DATA}/{fname}', newline='', encoding='utf-8-sig') as f:
        for row in csv.DictReader(f):
            text = row['Premises Addr.'].split('Census Tract')[0].strip().rstrip(',')
            a = split_one_line(text)
            if a:
                rid = f"{row['License Number'].strip()}-{row['License Type'].strip()}"
                record_addresses.append((source, rid, a))

with open(f'{DATA}/master-list-of-ca-licensed-contractors.csv',
          newline='', encoding='utf-8-sig') as f:
    for row in csv.DictReader(f):
        if row['MailingAddress'].strip():
            record_addresses.append(('CA-CONTRACTORS', row['LicenseNo'].strip(), {
                'street': row['MailingAddress'].strip(), 'city': row['City'].strip(),
                'region': row['State'].strip(), 'postal_code': row['ZIPCode'].strip()}))

# only the conference neighbourhood - the statewide SOS files hold ~10 local
# addresses between them and would burn the whole free tier on Sacramento
local = [(s, r, a) for s, r, a in record_addresses
         if a['city'].upper() in LOCAL_CITIES]

distinct = {}
for _, _, a in local:
    distinct[(a['street'].upper(), a['city'].upper())] = a

print(f"✅ {len(local):,} local record-addresses -> {len(distinct):,} distinct to geocode")


✅ 2,610 local record-addresses -> 2,076 distinct to geocode


In [4]:
from placekey.api import PlacekeyAPI
from placekey import placekey_to_geo

with psycopg2.connect(**PG) as conn, conn.cursor() as cur:
    cur.execute("SELECT street, city FROM workshop.geo_address")
    cached = {(s.upper(), c.upper()) for s, c in cur.fetchall()}

todo = {k: v for k, v in distinct.items() if k not in cached}
print(f"{len(cached):,} already cached, {len(todo):,} to look up")

if todo:
    pk_api = PlacekeyAPI(os.environ['PLACEKEY_API_KEY'])
    places = [{'query_id': f"{street}|{city}",
               'street_address': a['street'], 'city': a['city'],
               'region': a['region'], 'postal_code': a['postal_code'],
               'iso_country_code': 'US'}
              for (street, city), a in todo.items()]

    # batches of 100, self-throttling
    results = pk_api.lookup_placekeys(places, verbose=True)

    rows, failed = [], []
    for res in results:
        qid, pk = res.get('query_id'), res.get('placekey')
        if not qid or not pk:
            failed.append(res)
            continue
        street_key, city_key = qid.split('|', 1)
        a = todo[(street_key, city_key)]
        lat, lon = placekey_to_geo(pk)
        rows.append((a['street'], a['city'], a['region'], a['postal_code'], pk, lat, lon))

    with psycopg2.connect(**PG) as conn, conn.cursor() as cur:
        cur.executemany("""
            INSERT INTO workshop.geo_address
                   (street, city, region, postal_code, placekey, latitude, longitude)
            VALUES (%s, %s, %s, %s, %s, %s, %s)
            ON CONFLICT (street, city) DO NOTHING
        """, rows)

    print(f"✅ geocoded {len(rows):,}, failed {len(failed)}")
    for f in failed[:5]:
        print(f"   ✗ {f}")


0 already cached, 2,076 to look up


2026-09-08 20:02:11,450	INFO	Processed 1000 items
2026-09-08 20:03:16,769	INFO	Processed 2000 items
2026-09-08 20:04:20,103	INFO	Processed 2076 items
2026-09-08 20:04:20,104	INFO	Done


✅ geocoded 2,076, failed 0


In [7]:
import json, grpc, pandas as pd
from senzing import SzEngineFlags
from senzing_grpc import SzAbstractFactoryGrpc

grpc_url = f"{os.getenv('SENZING_GRPC_HOST', 'resolver')}:{os.getenv('SENZING_GRPC_PORT', '8261')}"
grpc_channel = grpc.insecure_channel(grpc_url)
grpc.channel_ready_future(grpc_channel).result(timeout=20)

sz_factory = SzAbstractFactoryGrpc(grpc_channel)
sz_engine = sz_factory.create_engine()
sz_configmanager = sz_factory.create_configmanager()

rows = []
handle = sz_engine.export_json_entity_report(SzEngineFlags.SZ_EXPORT_DEFAULT_FLAGS)
try:
    while True:
        line = sz_engine.fetch_next(handle)
        if not line:
            break
        resolved = json.loads(line)['RESOLVED_ENTITY']
        for rec in resolved.get('RECORDS', []):
            rows.append({'entity_id': resolved['ENTITY_ID'],
                         'entity_name': resolved.get('ENTITY_NAME', ''),
                         'data_source': rec['DATA_SOURCE'],
                         'record_id': rec['RECORD_ID']})
finally:
    sz_engine.close_export_report(handle)

df = pd.DataFrame(rows)
print(f"✅ {len(df):,} records in {df.entity_id.nunique():,} entities")


✅ 39,413 records in 31,339 entities


In [8]:
# df comes from the export cell above: entity_id / data_source / record_id
entity_by_record = {(r.data_source, r.record_id): r.entity_id for r in df.itertuples()}

with psycopg2.connect(**PG) as conn, conn.cursor() as cur:
    cur.execute("SELECT street, city, placekey, latitude, longitude FROM workshop.geo_address")
    geo = {(s.upper(), c.upper()): (s, c, pk, lat, lon)
           for s, c, pk, lat, lon in cur.fetchall()}

    seen, rows = set(), []
    for source, record_id, a in local:
        entity_id = entity_by_record.get((source, record_id))
        hit = geo.get((a['street'].upper(), a['city'].upper()))
        if entity_id is None or hit is None:
            continue
        street, city, pk, lat, lon = hit
        if (entity_id, street, city) in seen:
            continue
        seen.add((entity_id, street, city))
        rows.append((entity_id, street, city, pk, lat, lon))

    cur.execute("TRUNCATE workshop.entity_geo")      # rebuilt from scratch each run
    cur.executemany("""
        INSERT INTO workshop.entity_geo
               (entity_id, street, city, placekey, latitude, longitude)
        VALUES (%s, %s, %s, %s, %s, %s)
    """, rows)

    cur.execute("SELECT COUNT(*), COUNT(DISTINCT entity_id) FROM workshop.entity_geo")
    n_rows, n_entities = cur.fetchone()

    cur.execute("""
        SELECT n, COUNT(*) FROM (
            SELECT entity_id, COUNT(*) AS n FROM workshop.entity_geo GROUP BY entity_id
        ) s GROUP BY n ORDER BY n
    """)
    per_entity = cur.fetchall()

print(f"✅ {n_rows:,} geo rows covering {n_entities:,} entities "
      f"({n_entities / df.entity_id.nunique():.1%} of all {df.entity_id.nunique():,})")
print("\n   addresses per geo-enabled entity:")
for n, count in per_entity:
    print(f"     {n}: {count:,} entities")


✅ 2,482 geo rows covering 2,401 entities (7.7% of all 31,339)

   addresses per geo-enabled entity:
     1: 2,320 entities
     2: 81 entities


In [9]:
from pathlib import Path

OUT = Path('/workspace/data/geo')
OUT.mkdir(parents=True, exist_ok=True)

with psycopg2.connect(**PG) as conn:
    addresses = pd.read_sql("""
        SELECT street, city, region, postal_code, placekey, latitude, longitude
        FROM workshop.geo_address ORDER BY city, street
    """, conn)

addresses.to_csv(OUT / 'placekey_lookup.csv', index=False)
print(f"✅ wrote {len(addresses):,} addresses to {OUT / 'placekey_lookup.csv'}")


✅ wrote 2,076 addresses to /workspace/data/geo/placekey_lookup.csv


/tmp/ipykernel_115/1722852559.py:7: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  addresses = pd.read_sql("""
